# 🚀 AIC 2026: End-to-End Dual-GPU (2× T4) Production Pipeline (Scene-Adaptive Sampling)

This notebook implements the complete **AI Challenge (AIC) 2026** multi-modal retrieval pipeline:
1. **GPU 0**: SigLIP-SO400M @ 384px **scene-adaptive shot sampling** (eliminates redundant static frames)
2. **GPU 1**: Whisper large-v3 Vietnamese audio transcription & on-screen OCR
3. **Unified Indexing**: Scalable FAISS FlatIP vector index (~120k–400k frames) + Multi-modal BM25 lexical index
4. **Stage 2 Exact Localizer**: 30fps dense video decode around candidate timestamps (KIS & Q&A)
5. **TRAKE Stage 1**: DP-aligned video retrieval via scene index (Stage 2 VLM localization = future work)
6. **Submission Engine**: 100-rank portfolio optimization for competition metric $\frac{1}{5}\sum R@k$


In [ ]:
# 1. Verify Dual GPU Hardware (2x NVIDIA T4) & Setup Working Directory
import torch, os, sys

# Navigate into cloned repository if present
REPO_DIR = '/kaggle/working/aic2026'
if os.path.exists(REPO_DIR):
    os.chdir(REPO_DIR)
    print(f'Active working directory set to: {os.getcwd()}')

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

num_gpus = torch.cuda.device_count()
print(f'Detected {num_gpus} CUDA GPUs:')
for i in range(num_gpus):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)} (VRAM: {torch.cuda.get_device_properties(i).total_memory / 1e9:.2f} GB)')

assert num_gpus >= 1, 'Please enable GPU accelerator in Kaggle Settings (2x T4 recommended)!'

In [ ]:
# 2. Install Required Dependencies
!pip install -q open-clip-torch transformers faster-whisper openai-whisper faiss-cpu rank-bm25 deep-translator opencv-python easyocr fiftyone

In [ ]:
# 3. Symlink / Prepare Data Directories from Kaggle Input into aic2026/data
import os, glob

target_dir = os.path.join(os.getcwd(), 'data')
os.makedirs(target_dir, exist_ok=True)

if os.path.exists('/kaggle/input'):
    print(f'Detected Kaggle environment. Linking input datasets into {target_dir}...')
    for p in glob.glob('/kaggle/input/**/Videos_*', recursive=True):
        dest = os.path.join(target_dir, os.path.basename(p))
        if not os.path.exists(dest):
            os.symlink(p, dest)
    for p in glob.glob('/kaggle/input/**/Keyframes_*', recursive=True):
        dest = os.path.join(target_dir, os.path.basename(p))
        if not os.path.exists(dest):
            os.symlink(p, dest)
    for p in glob.glob('/kaggle/input/**/*media-info*', recursive=True):
        dest = os.path.join(target_dir, os.path.basename(p))
        if not os.path.exists(dest):
            os.symlink(p, dest)
    for p in glob.glob('/kaggle/input/**/*map-keyframes*', recursive=True):
        dest = os.path.join(target_dir, os.path.basename(p))
        if not os.path.exists(dest):
            os.symlink(p, dest)
    for p in glob.glob('/kaggle/input/**/*-aic25-b1', recursive=True):
        dest = os.path.join(target_dir, os.path.basename(p))
        if not os.path.exists(dest):
            os.symlink(p, dest)

print(f'Linked data contents ({target_dir}):', sorted(os.listdir(target_dir)))

### ⚡ Step 4: Parallel Dual-GPU Feature Extraction (Scene-Adaptive Video + Whisper ASR)

In [ ]:
import os
import subprocess

print('[Pipeline] Starting Dual-GPU Feature Extraction via isolated subprocesses (VRAM saturated)...')

repo_root = os.getcwd() if os.path.exists('src') else ('/kaggle/working/aic2026' if os.path.exists('/kaggle/working/aic2026') else '.')

# Environment isolation: GPU 0 for SigLIP vision, GPU 1 for Whisper ASR & OCR with PYTHONPATH
env_gpu0 = {**os.environ, 'CUDA_VISIBLE_DEVICES': '0', 'PYTHONUNBUFFERED': '1', 'PYTHONPATH': repo_root}
env_gpu1 = {**os.environ, 'CUDA_VISIBLE_DEVICES': '1', 'PYTHONUNBUFFERED': '1', 'PYTHONPATH': repo_root}

# SigLIP starts at batch 256, Whisper starts at batch 64 (both auto-halve on OOM)
p1 = subprocess.Popen(['python', 'scripts/extract_siglip_features.py', '--device', 'cuda:0', '--batch-size', '256'], env=env_gpu0, cwd=repo_root)
p2 = subprocess.Popen(['python', 'scripts/extract_whisper_asr.py', '--device', 'cuda:0', '--batch-size', '64'], env=env_gpu1, cwd=repo_root)

ret1 = p1.wait()
ret2 = p2.wait()
if ret1 != 0 or ret2 != 0:
    raise RuntimeError(f'GPU extraction failed (SigLIP exit: {ret1}, Whisper exit: {ret2})')

# Run OCR on GPU 1 after Whisper completes
p3 = subprocess.Popen(['python', 'scripts/extract_ocr.py', '--device', 'cuda:0'], env=env_gpu1, cwd=repo_root)
ret3 = p3.wait()
if ret3 != 0:
    raise RuntimeError(f'OCR extraction failed (Exit code: {ret3})')
print('✅ Dual-GPU extraction complete with 100% VRAM throughput!')

### 🏗️ Step 5: Build Scalable FAISS & Multi-Modal BM25 Indices

In [ ]:
# Build production FAISS index & unified lexical BM25 index
!python scripts/build_faiss_index.py

from src.index.metadata_indexer import MetadataIndexer
meta_idx = MetadataIndexer().build_and_cache(force=True)
print('Unified indices successfully built!')

### 🔍 Step 6: Interactive Multi-Modal Retrieval with Stage 2 Dense Localization

In [ ]:
import os, glob
import numpy as np
import torch
from src.index.faiss_index import FAISSIndex
from src.index.metadata_indexer import MetadataIndexer
from src.index.object_indexer import ObjectIndexer
from src.encoding.siglip_encoder import SigLIPEncoder
from src.query.text_encoder import CLIPTextEncoder
from src.query.translator import QueryTranslator
from src.retrieval.video_decoder import VideoDecoder
from src.evaluation.submission_generator import SubmissionGenerator

# Load FAISS Index and Lexical Indices
faiss_idx = FAISSIndex().load('cache/faiss_siglip')
meta_idx = MetadataIndexer().build_and_cache()
obj_idx = ObjectIndexer().build_and_cache()

# Select matching text encoder based on index dimensions
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
if faiss_idx.dim == 1152:
    print('Using SigLIP Text Encoder (1152-dim)...')
    text_encoder = SigLIPEncoder(device=device)
else:
    print('Using CLIP ViT-B/32 Text Encoder (512-dim)...')
    text_encoder = CLIPTextEncoder(device=device)

translator = QueryTranslator(use_online=True)
sub_gen = SubmissionGenerator(output_dir='submissions')
video_decoder = VideoDecoder(encoder=text_encoder, device=device)

def search_query(query_vi: str, top_k: int = 100, use_stage2_refinement: bool = True):
    en_query = translator.translate(query_vi)
    prompts = translator.generate_prompts(en_query)
    q_vec = text_encoder.encode_text(prompts, ensemble=True)
    
    # 1. FAISS Stage 1 dense candidate retrieval
    dense_results = faiss_idx.search(q_vec, top_k=top_k * 2)
    
    # 2. Hybrid re-ranking with Metadata BM25
    meta_scores = meta_idx.query(f'{query_vi} {en_query}', top_k=50)
    
    reranked = []
    for rec, score in dense_results:
        vid = rec['video_id']
        boost = 0.2 * (meta_scores.get(vid, 0.0) / max(1.0, max(meta_scores.values() or [1.0])))
        reranked.append((rec, score + boost))
        
    reranked.sort(key=lambda x: x[1], reverse=True)
    candidates = reranked[:top_k]
    
    # 3. Stage 2 Exact Localization for top-5 candidates (30fps continuous decode)
    if use_stage2_refinement:
        final_results = []
        for i, (rec, score) in enumerate(candidates):
            if i < 5:  # Refine top-5
                vid_p = glob.glob(f'data/**/video/{rec["video_id"]}.mp4', recursive=True)
                if vid_p and os.path.exists(vid_p[0]):
                    best_f, best_s, best_t = video_decoder.localize_exact_frame(
                        video_path=vid_p[0],
                        candidate_pts_time=rec['pts_time'],
                        query_vec=q_vec,
                        window_seconds=6.0
                    )
                    refined_rec = dict(rec)
                    refined_rec['frame_idx'] = best_f
                    refined_rec['pts_time'] = best_t
                    final_results.append((refined_rec, max(score, best_s)))
                    continue
            final_results.append((rec, score))
        return final_results
        
    return candidates

# Test sample query
sample_query = 'Người dẫn chương trình thời sự 60 giây trong trường quay'
results = search_query(sample_query, top_k=100, use_stage2_refinement=False)
print(f'Top 5 Results for "{sample_query}":')
for i, (rec, score) in enumerate(results[:5], 1):
    print(f'  [{i}] Video: {rec["video_id"]}, Frame: {rec["frame_idx"]}, Score: {score:.4f}')

### 📦 Step 7: Export Official Competition Submissions (Exact 100 Rows)

In [ ]:
# Generate official 100-row submission CSV and package bundle ZIP
sample_preds = [{'video_id': r[0]['video_id'], 'frame_idx': r[0]['frame_idx']} for r in results]
lines = sub_gen.format_kis_submission('query_01', sample_preds)
sub_gen.save_submission_file('query_01', lines)
zip_path = sub_gen.package_submission_zip('AIC2026_Submission_Bundle.zip')
print(f'Official Submission Package Ready: {zip_path} (Contains {len(lines)} rows in query_01.csv)')

### 📥 Step 8: Package & Download All Encoded Features (Visual, Audio ASR, OCR & FAISS Indices)

Compresses all generated `.npy` visual embeddings, Whisper transcripts, OCR text, and FAISS indices into a single downloadable ZIP archive in `/kaggle/working/`.

In [ ]:
import os, zipfile
from IPython.display import FileLink

zip_filename = 'aic2026_features_and_cache.zip'
zip_path = os.path.join('/kaggle/working' if os.path.exists('/kaggle/working') else '.', zip_filename)

print(f'[Export] Packaging cache directory into {zip_path}...')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk('cache'):
        for file in files:
            if file.endswith(('.npy', '.json', '.index', '.pkl')) and '.tmp.' not in file:
                abs_path = os.path.join(root, file)
                rel_path = os.path.relpath(abs_path, '.')
                zf.write(abs_path, arcname=rel_path)

zip_size_mb = os.path.getsize(zip_path) / (1024 * 1024)
print(f'✅ All features successfully packaged! Total Archive Size: {zip_size_mb:.2f} MB')
print(f'Artifact saved at: {zip_path}')
FileLink(zip_filename)